# Podocyte damage vs depletion: data audit notebook

**Question:** does a low podocyte damage score in a kidney biopsy mean the podocytes are damaged, or that there are fewer of them?

Run order: top to bottom. Use "Restart" then "Run All" to check that everything still works.
Environment: `conda activate podo`. Last updated: 2026-11-05.

**Contents**
1. Setup
2. PDS gene list (42 mouse genes)
3. ERCB (GSE104948): download sample table
4. ERCB: audit (platform, groups, centres)
5. NEPTUNE (GSE197307, GSE219185): audit
6. Series headers of all datasets (papers, files, overlaps)
7. NEPTUNE counts (GSE254957): peek and match samples to groups
8. Ortholog mapping, mouse to human (NOT RUN YET)
9. Next steps

## 1. Setup
Finds the repo root (the folder that contains `environment.yml`) and works from there, so paths such as `data/...` always work.
Run this cell first in every session.

In [1]:
import os, io, gzip, time, urllib.request, urllib.parse
from pathlib import Path

import pandas as pd
import GEOparse

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "environment.yml").exists()), None)
assert ROOT is not None, "environment.yml not found: open this notebook from inside the repo"
os.chdir(ROOT)
print("Working folder:", Path.cwd())

for d in ["data/raw/geo_meta", "data/metadata", "data/reference", "notes"]:
    Path(d).mkdir(parents=True, exist_ok=True)

Working folder: /home/intissar/projects/podocyte-damage-vs-depletion


## 2. PDS gene list
- Source: Padvitski et al., Cell Genomics 2026 (supplementary table).
- File: `data/reference/pds_genes.csv`, columns `mouse_gene_symbol` and `direction`.
- Checks: 42 genes, no duplicates, only "up"/"down" (expected: 39 down, 3 up).
- Issue found earlier and fixed here: a stray space in a column name, so spaces are stripped.

In [2]:
genes = pd.read_csv("data/reference/pds_genes.csv")
genes.columns = genes.columns.str.strip()
genes["mouse_gene_symbol"] = genes["mouse_gene_symbol"].str.strip()
genes["direction"] = genes["direction"].str.strip().str.lower()

assert len(genes) == 42, f"expected 42 genes, got {len(genes)}"
assert genes["mouse_gene_symbol"].is_unique, "duplicate gene symbols"
assert set(genes["direction"]) == {"down", "up"}, "unexpected direction values"
print(genes["direction"].value_counts().to_string())

genes.to_csv("data/reference/pds_genes.csv", index=False)   # clean header written back
mouse = genes["mouse_gene_symbol"].tolist()
print("OK:", len(mouse), "genes, e.g.", mouse[:5])

direction
down    39
up       3
OK: 42 genes, e.g. ['Aifm3', 'Angptl2', 'Astn2', 'C1qtnf7', 'Cers6']


## 3. ERCB (GSE104948): download the sample table
GEOparse in `how="brief"` mode returns the per-sample table (platform, tissue, diagnosis) but **not** series-level information (title, papers, files), so those come from section 6.
The raw GEO download goes to `data/raw/` (ignored by Git); the sample table goes to `data/metadata/` (committed).

In [3]:
def fetch_sample_table(acc):
    gse = GEOparse.get_GEO(geo=acc, destdir="data/raw/geo_meta", how="brief", silent=True)
    meta = gse.phenotype_data
    meta.to_csv(f"data/metadata/{acc}_metadata.csv")
    return meta

meta = fetch_sample_table("GSE104948")
print("samples x columns:", meta.shape)
print(meta[["title", "platform_id", "characteristics_ch1.0.tissue",
            "characteristics_ch1.1.diagnosis"]].head().to_string())

samples x columns: (196, 32)
                   title platform_id  characteristics_ch1.0.tissue characteristics_ch1.1.diagnosis
GSM2810645   H1-Glom-DN1    GPL24120  Glomeruli from kidney biopsy            Diabetic Nephropathy
GSM2810646  H1-Glom-DN10    GPL24120  Glomeruli from kidney biopsy            Diabetic Nephropathy
GSM2810647   H1-Glom-DN2    GPL24120  Glomeruli from kidney biopsy            Diabetic Nephropathy
GSM2810648   H1-Glom-DN5    GPL24120  Glomeruli from kidney biopsy            Diabetic Nephropathy
GSM2810649   H1-Glom-DN8    GPL24120  Glomeruli from kidney biopsy            Diabetic Nephropathy


## 4. ERCB audit
Findings recorded here:
- 196 glomerular biopsies on two platforms: GPL22945 (71) and GPL24120 (125).
- **No eGFR, proteinuria, age or sex** in the public metadata.
- 21 samples have an **empty diagnosis**: these are the living donors, labelled `LD` only in the sample title.
- **Platform = centre:** site H7 is entirely GPL22945, sites H1-H5 entirely GPL24120. Compare groups **within one platform only**.
- Controls: GPL22945 has 18 living donors + 3 tumor nephrectomy; GPL24120 has only 3 + 2.
- Samples also exist in older series GSE47183 and GSE32591: do not combine.

In [5]:
meta = pd.read_csv("data/metadata/GSE104948_metadata.csv", index_col=0)

# group and centre are encoded in the sample title, e.g. "H7-Glom-LD_988"
t = meta["title"].str.strip()
meta["site"] = t.str.split("-").str[0]
meta["group"] = t.str.split("-").str[2].str.replace(r"[_\d]+$", "", regex=True)

lines = []
lines.append(f"samples: {len(meta)}")
lines.append("\nPLATFORM:\n" + meta["platform_id"].value_counts().to_string())
lines.append("\nTISSUE:\n" + meta["characteristics_ch1.0.tissue"].value_counts().to_string())
lines.append("\nDIAGNOSIS column (incl. missing):\n"
             + meta["characteristics_ch1.1.diagnosis"].value_counts(dropna=False).to_string())
lines.append("\nGROUP (from title) x PLATFORM:\n"
             + pd.crosstab(meta["group"], meta["platform_id"]).to_string())
lines.append("\nSITE x PLATFORM:\n"
             + pd.crosstab(meta["site"], meta["platform_id"]).to_string())

kw = ["gfr", "creat", "protein", "albumin", "age", "sex", "gender", "follow", "outcome", "esrd"]
hits = [c for c in meta.columns if any(k in c.lower() for k in kw)]
lines.append("\nPossible clinical columns: " + str(hits))

text = "\n".join(lines)
Path("notes/geo_audit_GSE104948.txt").write_text(text)
meta.to_csv("data/metadata/GSE104948_metadata_parsed.csv")
print(text)

samples: 196

PLATFORM:
platform_id
GPL24120    125
GPL22945     71

TISSUE:
characteristics_ch1.0.tissue
Glomeruli from kidney biopsy    196

DIAGNOSIS column (incl. missing):
characteristics_ch1.1.diagnosis
Systemic Lupus Erythematosus                                   32
IgA Nephropathy                                                27
ANCA Associated Vasculitis                                     22
NaN                                                            21
Membranous Glomerulonephropathy                                21
Focal Segmental Glomerular Sclerosis                           18
Hypertensive Nephropathy                                       15
Minimal Change Disease                                         14
Diabetic Nephropathy                                           12
Focal Segmental Glomerular Sclerosis/Minimal Change Disease     6
Tumor Nephrectomy                                               5
Thin Membrande Disease                                          3

## 5. NEPTUNE audit (GSE197307 and GSE219185)
- **GSE197307:** glomeruli, 282 samples (FSGS 93, MCD 89, membranous 62, other nephrotic syndrome 29, healthy donor 8, unspecified 1). Superseded by GSE254957.
- **GSE219185:** **tubulointerstitium**, not glomeruli (220 samples: FSGS 114, MCD 106). Not independent of other NEPTUNE sets. Negative control only.
- No clinical variables (eGFR, proteinuria) in either series.
- GSE254957 has **no sample table** in GEO (see section 7).

In [6]:
def audit(acc):
    meta = fetch_sample_table(acc)
    L = [f"=== {acc}: {meta.shape[0]} samples, {meta.shape[1]} columns"]
    L.append("platform: " + str(meta["platform_id"].value_counts().to_dict()))
    L.append("titles: " + "; ".join(meta["title"].head(5)))
    for c in [c for c in meta.columns if c.startswith("characteristics")]:
        vc = meta[c].value_counts(dropna=False)
        L.append(f"- {c} ({len(vc)} unique): "
                 + "; ".join(f"{k} [{v}]" for k, v in vc.head(8).items()))
    kw = ["gfr", "creat", "protein", "upcr", "albumin", "age", "sex", "gender",
          "race", "follow", "outcome", "esrd"]
    L.append("possible clinical columns: "
             + str([c for c in meta.columns if any(k in c.lower() for k in kw)]))
    return "\n".join(L)

text = "\n\n".join(audit(a) for a in ["GSE219185", "GSE197307"])
Path("notes/geo_audit_neptune.txt").write_text(text)
print(text)

=== GSE219185: 220 samples, 34 columns
platform: {'GPL18573': 220}
titles: Focal Segmental Glomerulosclerosis [Sample_1]; Focal Segmental Glomerulosclerosis [Sample_10]; Minimal Change Disease [Sample_100]; Focal Segmental Glomerulosclerosis [Sample_101]; Minimal Change Disease [Sample_102]
- characteristics_ch1.0.tissue (1 unique): Tubulointerstitium from human kidney biopsy [220]
- characteristics_ch1.1.disease (2 unique): Focal Segmental Glomerulosclerosis [114]; Minimal Change Disease [106]
possible clinical columns: []

=== GSE197307: 282 samples, 32 columns
platform: {'GPL18573': 282}
titles: Sample_1_LivingDonor; Sample_2_LivingDonor; Sample_3_LivingDonor; Sample_4_LivingDonor; Sample_5_LivingDonor
- characteristics_ch1.0.tissue (1 unique): Glomeruli from kidney biopsy [282]
- characteristics_ch1.1.disease (6 unique): Focal Segmental Glomerulosclerosis [93]; Minimal Change Disease [89]; Membranous Nephropathy [62]; Other Nephrotic Syndrome [29]; Healthy living transplant donor [

## 6. Series headers (papers, files, overlaps)
Read directly from GEO as text, with automatic retries (NCBI sometimes returns a temporary 502 error).
What this showed:
- **GSE254957 is a re-analysis of GSE197307** (same 282 samples): use GSE254957, never both.
- GSE219185 shares samples with other NEPTUNE sets.
- GSE104948 (ERCB) samples were also deposited as GSE47183 and GSE32591.
- Raw data and clinical data for NEPTUNE need an approved ancillary study request.
- Papers: ERCB 29724730, 40059827; NEPTUNE 35477557, 36442540, 38838223, 41659822 (titles still to be looked up in PubMed).

*Dead end removed from the old notebook:* reading these fields through `GEOparse` (`gse.metadata`) returned empty values in `brief` mode.

In [7]:
def get_header(acc, tries=4, wait=8):
    url = ("https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi"
           f"?acc={acc}&targ=self&form=text&view=brief")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    for i in range(1, tries + 1):
        try:
            return urllib.request.urlopen(req, timeout=60).read().decode("utf-8", "replace")
        except Exception as e:
            print(f"  {acc}: attempt {i}/{tries} failed ({e})")
            time.sleep(wait * i)
    return None

keep = ("!Series_title", "!Series_pubmed_id", "!Series_supplementary_file",
        "!Series_summary", "!Series_overall_design", "!Series_relation")

lines = []
for acc in ["GSE254957", "GSE197307", "GSE219185", "GSE104948"]:
    lines.append("=" * 60 + "\n" + acc)
    txt = get_header(acc)
    if txt is None:
        lines.append("FAILED after retries")
        continue
    for line in txt.splitlines():
        if line.startswith(keep):
            lines.append(line[:1200])

text = "\n".join(lines)
Path("notes/geo_series_headers.txt").write_text(text)
print(text)

GSE254957
!Series_title = Glomerular voom-transformed RNAseq counts from the Neptune Cohort and healthy living donors [re-analysis of GSE197307]
!Series_pubmed_id = 36442540
!Series_pubmed_id = 41659822
!Series_summary = Glomerular Transcriptome in the Nephrotic Syndrome study network cohort
!Series_overall_design = Please note that GSE197307 samples were re-analyzed as described below.
!Series_overall_design = Glomerular RNAseq profiles; Raw data (fastq) files are available upon approved ancillary study request from the NEPTUNE consortium (please see https://www.neptune-study.org/ancillary-studies; email contact: NEPTUNE-STUDY@umich.edu
!Series_overall_design = *data processing step:
!Series_overall_design = fastqScreen 0.11.4
!Series_overall_design = cutadapt 1.14 -m 50 -A CTGTCTCTTATA -a CTGTCTCTTATA
!Series_overall_design = STAR 2.5.2.a --bamRemoveDuplicatesType UniqueIdentical --twopassMode Basic --outFilterIntronMotifs RemoveNoncanonical --outSAMtype BAM SortedByCoordinate --outS

## 7. NEPTUNE counts (GSE254957): peek and match samples to groups
GSE254957 has no per-sample records, so:
1. read only the **header line** of the counts file (streamed, not the whole file);
2. match the sample names to the GSE197307 sample table to get each sample's disease group.

Findings: 283 header fields = 1 gene column + **282 samples**; genes are **Ensembl IDs**; values are **raw integer counts**; donor names contain the group, patient samples are bare `Sample_N` and need the GSE197307 table.
Result: 282 of 282 matched. Output: `data/metadata/GSE254957_samples_from_GSE197307.csv`.

In [8]:
url = ("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE254nnn/GSE254957/suppl/"
       "GSE254957_NEPTUNE_GSE197307_UpdatedCountData.txt.gz")
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=60) as resp:
    with gzip.GzipFile(fileobj=resp) as gz:
        header = gz.readline().decode("utf-8", "replace").rstrip("\n").split("\t")
        first = gz.readline().decode("utf-8", "replace").rstrip("\n").split("\t")

print("header fields:", len(header), "| first:", header[:3], "| last:", header[-3:])
print("first data row:", first[:6])

m197 = pd.read_csv("data/metadata/GSE197307_metadata.csv", index_col=0)
ids_matrix = [h.strip() for h in header[1:]]
ids_geo = m197["title"].str.strip().tolist()

only_matrix = sorted(set(ids_matrix) - set(ids_geo))
only_geo = sorted(set(ids_geo) - set(ids_matrix))
print("in matrix, not in GEO:", only_matrix[:10])
print("in GEO, not in matrix:", only_geo[:10])
assert not only_matrix and not only_geo, "sample names do not match"

samples = (m197.assign(sample_id=m197["title"].str.strip())
                [["sample_id", "characteristics_ch1.1.disease"]]
                .rename(columns={"characteristics_ch1.1.disease": "disease"}))
samples = samples[samples["sample_id"].isin(ids_matrix)]
samples.to_csv("data/metadata/GSE254957_samples_from_GSE197307.csv", index=False)
print("\nsaved", len(samples), "samples")
print(samples["disease"].value_counts().to_string())

header fields: 283 | first: ['ENSEMBL_gene_ID', 'Sample_1_LivingDonor', 'Sample_2_LivingDonor'] | last: ['Sample_319', 'Sample_337', 'Sample_339']
first data row: ['ENSG00000000003', '3767', '3707', '5463', '3023', '3639']
in matrix, not in GEO: []
in GEO, not in matrix: []

saved 282 samples
disease
Focal Segmental Glomerulosclerosis    93
Minimal Change Disease                89
Membranous Nephropathy                62
Other Nephrotic Syndrome              29
Healthy living transplant donor        8
Nephrotic Syndrome, unspecified        1


## 8. Ortholog mapping (mouse to human):
The score was built in mice; our data are human, so each of the 42 mouse genes needs a human version.
- Source: Ensembl BioMart. **Write the Ensembl release/date into `notes/log.md`.**
- Keep **one-to-one** orthologs; list genes with several or no matches.
- NEPTUNE needs the **Ensembl ID**, ERCB needs the **Entrez ID**; keep the human symbol too.
- Report how many of the 42 are usable (goes into the README and the pre-registration).
- Output: `data/reference/pds_orthologs_raw.csv`, then a cleaned table.

In [9]:
def biomart(xml, tries=3):
    url = "https://www.ensembl.org/biomart/martservice?query=" + urllib.parse.quote(xml)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    for i in range(tries):
        try:
            txt = urllib.request.urlopen(req, timeout=120).read().decode()
            return pd.read_csv(io.StringIO(txt), sep="\t")
        except Exception as e:
            print("retry", i + 1, e)
            time.sleep(8 * (i + 1))
    raise RuntimeError("BioMart failed")

query = """<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE Query>
<Query virtualSchemaName="default" formatter="TSV" header="1" uniqueRows="1" count="" datasetConfigVersion="0.6">
<Dataset name="mmusculus_gene_ensembl" interface="default">
<Filter name="external_gene_name" value="%s"/>
<Attribute name="external_gene_name"/>
<Attribute name="ensembl_gene_id"/>
<Attribute name="hsapiens_homolog_ensembl_gene"/>
<Attribute name="hsapiens_homolog_associated_gene_name"/>
<Attribute name="hsapiens_homolog_orthology_type"/>
<Attribute name="hsapiens_homolog_orthology_confidence"/>
</Dataset></Query>""" % ",".join(mouse)

orth = biomart(query)
orth.columns = ["mouse_symbol", "mouse_ensembl", "human_ensembl", "human_symbol",
                "orthology_type", "confidence"]
orth.to_csv("data/reference/pds_orthologs_raw.csv", index=False)

print(orth.shape)
print(orth["orthology_type"].value_counts(dropna=False).to_string())
print("\nNot returned at all:", sorted(set(mouse) - set(orth["mouse_symbol"])))

(42, 6)
orthology_type
ortholog_one2one    42

Not returned at all: []


## 8b. Check the ortholog table
All 42 mouse genes returned a one-to-one human match. Check by eye: do the names look sensible (for example Wt1 to WT1, Zfp423 to ZNF423), is anything missing or shared, and is the confidence high?

In [10]:
orth = pd.read_csv("data/reference/pds_orthologs_raw.csv")
tab = genes.merge(orth, left_on="mouse_gene_symbol", right_on="mouse_symbol", how="left")

print("rows:", len(tab))
print("missing human Ensembl ID:", tab["human_ensembl"].isna().sum())
print("human genes hit by more than one mouse gene:", tab["human_ensembl"].duplicated(keep=False).sum())
print("confidence values:", tab["confidence"].value_counts(dropna=False).to_dict())
print(tab[["mouse_gene_symbol", "direction", "human_symbol",
           "human_ensembl", "confidence"]].to_string())

rows: 42
missing human Ensembl ID: 0
human genes hit by more than one mouse gene: 0
confidence values: {1: 42}
   mouse_gene_symbol direction human_symbol    human_ensembl  confidence
0              Aifm3      down        AIFM3  ENSG00000183773           1
1            Angptl2      down      ANGPTL2  ENSG00000136859           1
2              Astn2      down        ASTN2  ENSG00000148219           1
3            C1qtnf7      down      C1QTNF7  ENSG00000163145           1
4              Cers6      down        CERS6  ENSG00000172292           1
5              Cldn5      down        CLDN5  ENSG00000184113           1
6              Clic3      down        CLIC3  ENSG00000169583           1
7              Dach1      down        DACH1  ENSG00000276644           1
8                Ddn      down          DDN  ENSG00000181418           1
9             Dusp15      down       DUSP15  ENSG00000149599           1
10             Enpep      down        ENPEP  ENSG00000138792           1
11           

## 8c. Add Entrez IDs
NEPTUNE uses Ensembl IDs; ERCB uses Entrez IDs. Ask BioMart for the Entrez ID of each human gene.

In [11]:
ens_list = ",".join(tab["human_ensembl"].dropna().unique())
query2 = """<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE Query>
<Query virtualSchemaName="default" formatter="TSV" header="1" uniqueRows="1" count="" datasetConfigVersion="0.6">
<Dataset name="hsapiens_gene_ensembl" interface="default">
<Filter name="ensembl_gene_id" value="%s"/>
<Attribute name="ensembl_gene_id"/>
<Attribute name="entrezgene_id"/>
</Dataset></Query>""" % ens_list

ent = biomart(query2)
ent.columns = ["human_ensembl", "human_entrez"]
print("rows returned:", len(ent), "| Ensembl IDs with several Entrez IDs:",
      ent["human_ensembl"].duplicated().sum())
print("without Entrez ID:", ent["human_entrez"].isna().sum())

ent = ent.dropna().drop_duplicates("human_ensembl")   # keep the first Entrez ID per gene
ent["human_entrez"] = ent["human_entrez"].astype("Int64")
tab = tab.drop(columns=["human_entrez"], errors="ignore").merge(ent, on="human_ensembl", how="left")
print(tab[["human_symbol", "human_ensembl", "human_entrez"]].head(10).to_string())

rows returned: 42 | Ensembl IDs with several Entrez IDs: 0
without Entrez ID: 0
  human_symbol    human_ensembl  human_entrez
0        AIFM3  ENSG00000183773        150209
1      ANGPTL2  ENSG00000136859         23452
2        ASTN2  ENSG00000148219         23245
3      C1QTNF7  ENSG00000163145        114905
4        CERS6  ENSG00000172292        253782
5        CLDN5  ENSG00000184113          7122
6        CLIC3  ENSG00000169583          9022
7        DACH1  ENSG00000276644          1602
8          DDN  ENSG00000181418         23109
9       DUSP15  ENSG00000149599        128853


## 8d. Gene coverage per dataset
Downloads the NEPTUNE counts file (about tens of MB) and the two ERCB matrices into `data/raw/` (ignored by Git), reads only the gene ID column, and checks which of our genes are measured. Saves the final working table `data/reference/pds_genes_human.csv`.

In [12]:
import shutil

def download(url, dest, tries=3):
    if Path(dest).exists():
        return
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    for i in range(1, tries + 1):
        try:
            with urllib.request.urlopen(req, timeout=300) as r, open(dest, "wb") as f:
                shutil.copyfileobj(r, f)
            return
        except Exception as e:
            print("retry", i, e)
            time.sleep(8 * i)
    raise RuntimeError("download failed: " + url)

# NEPTUNE (Ensembl IDs)
nep_path = "data/raw/GSE254957_counts.txt.gz"
download("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE254nnn/GSE254957/suppl/"
         "GSE254957_NEPTUNE_GSE197307_UpdatedCountData.txt.gz", nep_path)
nep_ids = (pd.read_csv(nep_path, sep="\t", usecols=[0]).iloc[:, 0]
             .astype(str).str.replace(r"\.\d+$", "", regex=True))
print("NEPTUNE genes measured:", len(nep_ids), "| example:", nep_ids.head(3).tolist())
tab["in_neptune"] = tab["human_ensembl"].isin(set(nep_ids))

# ERCB (Entrez IDs, one matrix per platform)
tab["human_entrez_str"] = tab["human_entrez"].astype("string")
for gpl in ["GPL22945", "GPL24120"]:
    p = f"data/raw/GSE104948-{gpl}_series_matrix.txt.gz"
    download("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE104nnn/GSE104948/matrix/"
             f"GSE104948-{gpl}_series_matrix.txt.gz", p)
    ids = (pd.read_csv(p, sep="\t", comment="!", usecols=[0]).iloc[:, 0]
             .astype(str).str.replace(r"_at$", "", regex=True))
    print(gpl, "probe sets:", len(ids), "| example IDs:", ids.head(3).tolist())
    tab[f"in_ercb_{gpl}"] = tab["human_entrez_str"].isin(set(ids))

cols = ["mouse_gene_symbol", "direction", "human_symbol", "human_ensembl", "human_entrez",
        "in_neptune", "in_ercb_GPL22945", "in_ercb_GPL24120"]
final = tab[cols]
final.to_csv("data/reference/pds_genes_human.csv", index=False)

print("\nUsable genes (of 42):")
print(final[["in_neptune", "in_ercb_GPL22945", "in_ercb_GPL24120"]].sum().to_string())
print("\nNot found in at least one dataset:")
print(final[~final[["in_neptune", "in_ercb_GPL22945", "in_ercb_GPL24120"]].all(axis=1)].to_string())

NEPTUNE genes measured: 51534 | example: ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419']
GPL22945 probe sets: 12074 | example IDs: ['10000', '10001', '10002']
GPL24120 probe sets: 12074 | example IDs: ['10000', '10001', '10002']

Usable genes (of 42):
in_neptune          42
in_ercb_GPL22945    34
in_ercb_GPL24120    34

Not found in at least one dataset:
   mouse_gene_symbol direction human_symbol    human_ensembl  human_entrez  in_neptune  in_ercb_GPL22945  in_ercb_GPL24120
0              Aifm3      down        AIFM3  ENSG00000183773        150209        True             False             False
3            C1qtnf7      down      C1QTNF7  ENSG00000163145        114905        True             False             False
9             Dusp15      down       DUSP15  ENSG00000149599        128853        True             False             False
18            Metrnl      down       METRNL  ENSG00000176845        284207        True             False             False
24            Pard

## 10. Load NEPTUNE counts and check data quality (no group comparisons)
Loads the GSE254957 counts (genes x samples), strips Ensembl version suffixes, aligns samples to the label table, and checks that values are integer counts. Looks only at overall depth and gene detection, never at groups or scores. Saves a processed copy to `data/processed/` (not committed to Git).

In [3]:
import numpy as np

counts = pd.read_csv("data/raw/GSE254957_counts.txt.gz", sep="\t", index_col=0)
counts.index = counts.index.astype(str).str.replace(r"\.\d+$", "", regex=True)

samples = pd.read_csv("data/metadata/GSE254957_samples_from_GSE197307.csv").set_index("sample_id")
assert set(counts.columns) == set(samples.index), "sample names do not match"
counts = counts[samples.index]                      # same order as the label table

print("genes x samples:", counts.shape)
print("gene IDs unique:", counts.index.is_unique)

vals = counts.to_numpy()
print("all integer-valued:", bool(np.all(np.mod(vals, 1) == 0)), "| min value:", vals.min())
print("genes with zero counts in every sample:", int((counts.sum(axis=1) == 0).sum()))

lib = counts.sum(axis=0)                            # overall depth per sample, NOT by group
print("\nlibrary size (all samples together):")
print(lib.describe().round(0).to_string())

# detection of the 42 score genes across all samples (technical check only)
pds = pd.read_csv("data/reference/pds_genes_human.csv")
det = (counts.reindex(pds["human_ensembl"]) > 0).sum(axis=1)
pds["samples_detected"] = det.to_numpy()
print("\nPDS genes detected in the fewest samples:")
print(pds.sort_values("samples_detected")[["human_symbol", "samples_detected"]].head(8).to_string(index=False))

Path("data/processed").mkdir(parents=True, exist_ok=True)
counts.astype("int32").to_parquet("data/processed/neptune_glom_counts.parquet")
samples.to_csv("data/processed/neptune_glom_samples.csv")
print("\nsaved processed files")

genes x samples: (51534, 282)
gene IDs unique: True
all integer-valued: True | min value: 0
genes with zero counts in every sample: 3224

library size (all samples together):
count         282.0
mean     36810671.0
std       6232997.0
min      22497620.0
25%      32814726.0
50%      36054446.0
75%      39573750.0
max      67498182.0

PDS genes detected in the fewest samples:
human_symbol  samples_detected
      FGFBP1               109
      SEMA3E               240
       EPHB1               241
      ANKRD1               246
       AIFM3               256
      SNCAIP               276
       CLIC3               279
      DUSP15               281


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip, conda, or your preferred package management tool to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip, conda, or your preferred package management tool to install the fastparquet package.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# go to the project root (the folder that contains environment.yml)
ROOT = Path.cwd()
while not (ROOT / "environment.yml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Working in:", Path.cwd())

Working in: /home/intissar/projects/podocyte-damage-vs-depletion


In [4]:
#2. Count how many samples the exclusion rule would remove. Run this in a new cell:
lib = counts.sum(axis=0)
loglib = np.log10(lib)
med = loglib.median()
mad = (loglib - med).abs().median() * 1.4826     # scaled MAD, comparable to a standard deviation
z = (loglib - med) / mad
flag = z.abs() > 3
print("MAD-scaled threshold: 3 | samples flagged:", int(flag.sum()))
print(lib[flag].round(0).to_string())

MAD-scaled threshold: 3 | samples flagged: 8
Sample_298    22497620
Sample_244    65463110
Sample_264    58573578
Sample_336    56864472
Sample_110    67498182
Sample_270    23545323
Sample_326    58049246
Sample_179    55475459


## 11. Load ERCB (GSE104948, GPL22945) and check data quality (no group comparisons)
Reads the series matrix (probe sets x samples, Entrez IDs), matches samples to the parsed metadata, checks the value scale, and confirms the 34 common PDS genes are present. Counts group sizes only. No scores, no group comparisons.

In [5]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "environment.yml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

# 1. expression table: lines starting with "!" are header text, so we skip them
expr = pd.read_csv("data/raw/GSE104948-GPL22945_series_matrix.txt.gz",
                   sep="\t", comment="!", index_col=0)
expr.index = expr.index.astype(str)
print("probe sets x samples:", expr.shape)
print("IDs unique:", expr.index.is_unique)
print("missing values:", int(expr.isna().sum().sum()))

# 2. value scale: raw intensities (large numbers) or log2 (about 2 to 16)?
v = expr.to_numpy(dtype=float)
print("min / median / max:", np.nanmin(v).round(2), np.nanmedian(v).round(2), np.nanmax(v).round(2))

# 3. metadata: find the column that holds the sample names
meta = pd.read_csv("data/metadata/GSE104948_metadata_parsed.csv")
print("\nmetadata columns:", meta.columns.tolist())
id_col = max(meta.columns, key=lambda c: meta[c].astype(str).isin(expr.columns).sum())
print("sample-name column:", id_col, "| matches:", int(meta[id_col].astype(str).isin(expr.columns).sum()),
      "of", expr.shape[1], "expression samples")
meta = meta.set_index(id_col)

# 4. design counts only (how many samples per site and group)
common = [c for c in expr.columns if c in meta.index]
print("\nsamples with metadata:", len(common))
print(meta.loc[common].groupby(["site", "group"]).size().to_string())

# 5. are the 34 common PDS genes present?
pds = pd.read_csv("data/reference/pds_genes_human.csv")
have = pds[pds["in_ercb_GPL22945"]].copy()
have["present"] = have["human_entrez"].astype(str).isin(expr.index)
print("\ncommon PDS genes found by Entrez ID:", int(have["present"].sum()), "of", len(have))

# 6. save
Path("data/processed").mkdir(parents=True, exist_ok=True)
expr[common].to_parquet("data/processed/ercb_gpl22945_expr.parquet")
meta.loc[common].to_csv("data/processed/ercb_gpl22945_samples.csv")
print("\nsaved processed files")

probe sets x samples: (12074, 71)
IDs unique: True
missing values: 0
min / median / max: 3.25 7.24 14.44

metadata columns: ['Unnamed: 0', 'title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'taxid_ch1', 'characteristics_ch1.0.tissue', 'characteristics_ch1.1.diagnosis', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1', 'label_protocol_ch1', 'hyb_protocol', 'scan_protocol', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_institute', 'contact_address', 'contact_city', 'contact_state', 'contact_zip/postal_code', 'contact_country', 'supplementary_file', 'series_id', 'data_row_count', 'site', 'group']
sample-name column: Unnamed: 0 | matches: 71 of 71 expression samples

samples with metadata: 71
site  group   
H7    DN           7
      FSGS        10
      FSGS&MCD     6
      LD          18
      MCD          5
      RPGN        22
      TN           3

common 

In [6]:
pds = pd.read_csv("data/reference/pds_genes_human.csv")
have = pds[pds["in_ercb_GPL22945"]].copy()
have["entrez_id"] = have["human_entrez"].astype("Int64").astype(str)
have["present"] = have["entrez_id"].isin(expr.index)
print("common PDS genes found:", int(have["present"].sum()), "of", len(have))
print(have.loc[~have["present"], ["human_symbol", "entrez_id"]])

common PDS genes found: 0 of 34
   human_symbol entrez_id
1       ANGPTL2     23452
2         ASTN2     23245
4         CERS6    253782
5         CLDN5      7122
6         CLIC3      9022
7         DACH1      1602
8           DDN     23109
10        ENPEP      2028
11        EPHB1      2047
12       FGFBP1      9982
13         GJA3      2700
14        ITGB8      3696
15        LRRC1     55227
16         MAFB      9935
17        MAGI2      9863
19        MTSS1      9788
20       NAP1L1      4673
21         NEK1      4750
22         OPTN     10133
23         PAK1      5058
25        PLCE1     51196
26        PTPRO      5800
28       SEMA3E      9723
29       SEMA3G     56920
31      SMARCA2      6595
32       SNCAIP      9627
33      SPATS2L     26010
34        SULF1     23213
35       THSD7A    221981
36      TSPAN15     23555
37          WT1      7490
38       ZNF423     23090
39       ANKRD1     27063
40          F2R      2149


In [7]:
print("first row labels:", expr.index[:8].tolist())
print("last row labels: ", expr.index[-3:].tolist())

# try to pull the number out of each label (works for '10000', '10000_at', 'ENTREZ_10000', ...)
expr_ids = expr.index.str.extract(r"(\d+)")[0]
print("labels with a number:", int(expr_ids.notna().sum()), "of", len(expr_ids))

have["present2"] = have["entrez_id"].isin(set(expr_ids.dropna()))
print("common PDS genes found after extraction:", int(have["present2"].sum()), "of", len(have))

first row labels: ['10000_at', '10001_at', '10002_at', '10003_at', '100048912_at', '10004_at', '10005_at', '10006_at']
last row labels:  ['9997_at', '999_at', '9_at']
labels with a number: 12074 of 12074
common PDS genes found after extraction: 34 of 34


## 10b. NEPTUNE: flag library-size outliers (depth only, no groups)
Counts how many samples have an unusually low or high sequencing depth (more than 3 scaled MADs from the median of log10 library size). Informational only: the primary analysis keeps all samples, and the flagged ones are used in a sensitivity analysis.

In [8]:
lib = counts.sum(axis=0)
loglib = np.log10(lib)
med = loglib.median()
mad = (loglib - med).abs().median() * 1.4826
z = (loglib - med) / mad
flag = z.abs() > 3
print("Samples flagged:", int(flag.sum()))
print(lib[flag].round(0).to_string())

Samples flagged: 8
Sample_298    22497620
Sample_244    65463110
Sample_264    58573578
Sample_336    56864472
Sample_110    67498182
Sample_270    23545323
Sample_326    58049246
Sample_179    55475459


## 10c. Direction of FGFBP1 (the weakly detected score gene)
FGFBP1 is detected in only 109 of 282 NEPTUNE samples. This shows whether it is an "up" or a "down" gene in the score, for the sensitivity analysis that removes it.

In [9]:
print(pds.loc[pds.human_symbol == "FGFBP1", ["human_symbol", "direction"]])

   human_symbol direction
12       FGFBP1      down


## 11b. ERCB: clean the probe labels to plain Entrez IDs
The series matrix labels look like `10000_at`. This extracts the number so the table can be matched to the PDS gene list, checks that the IDs are unique, confirms all 34 common genes are present, and re-saves the processed table.

In [10]:
expr.index = expr.index.str.extract(r"(\d+)")[0].to_numpy()
print("IDs unique after cleaning:", expr.index.is_unique)

have["present"] = have["entrez_id"].isin(expr.index)
print("common PDS genes found:", int(have["present"].sum()), "of", len(have))

expr[common].to_parquet("data/processed/ercb_gpl22945_expr.parquet")
print("re-saved with clean Entrez IDs")

IDs unique after cleaning: True
common PDS genes found: 34 of 34
re-saved with clean Entrez IDs


## 12a. Create the folder for our own code
Makes `src/`, where reusable functions live (so notebooks stay short). Also moves into the project root in case the kernel was restarted.

In [11]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "environment.yml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
Path("src").mkdir(exist_ok=True)
print("Working in:", Path.cwd(), "| src exists:", Path("src").exists())

Working in: /home/intissar/projects/podocyte-damage-vs-depletion | src exists: True


## 12b. Scoring function (PROVISIONAL: standard AUCell logic, to be validated against the authors' code)
Writes `src/pds_score.py`. Steps: rank genes in each sample (rank 1 = highest expression); the AUC of a gene set counts how many of its genes sit within the top 5% of ranks, weighting higher ranks more; then DS = (n_up/N)*AUC_up - (n_down/N)*AUC_down, so a HIGHER DS means MORE damage. Optionally scaled (z-score) within the cohort.

In [12]:
%%writefile src/pds_score.py
import numpy as np
import pandas as pd


def auc_score(expr, genes, auc_max_frac=0.05):
    """AUC of a gene set per sample. expr: genes x samples (index = gene IDs)."""
    ranks = expr.rank(axis=0, ascending=False, method="average")   # 1 = highest
    max_rank = int(np.ceil(auc_max_frac * expr.shape[0]))
    sub = ranks.loc[genes]
    contrib = (max_rank - sub).clip(lower=0)                        # only genes inside the top max_rank count
    return contrib.sum(axis=0) / (len(genes) * max_rank)


def pds_score(expr, up, down, scale=True):
    """Podocyte damage score: HIGHER = MORE damage.
    DS = (n_up/N)*AUC_up - (n_down/N)*AUC_down, N = n_up + n_down (genes present)."""
    up = [g for g in up if g in expr.index]
    down = [g for g in down if g in expr.index]
    n_up, n_down = len(up), len(down)
    n_total = n_up + n_down
    ds_up = auc_score(expr, up) if n_up else 0.0
    ds_down = auc_score(expr, down) if n_down else 0.0
    ds = (n_up / n_total) * ds_up - (n_down / n_total) * ds_down
    if scale:
        ds = (ds - ds.mean()) / ds.std(ddof=1)
    return ds

Writing src/pds_score.py


## 12c. Toy test with invented data (no real samples)
Builds 2,000 fake genes x 60 fake samples. 39 "down" genes and 3 "up" genes are planted. Healthy samples keep the baseline; damaged samples have lower "down" genes and higher "up" genes. Checks that the score is finite, and clearly higher in damaged samples.

In [13]:
import importlib, sys
sys.path.insert(0, "src")
import pds_score
importlib.reload(pds_score)

rng = np.random.default_rng(20261106)
n_genes, n_per_group = 2000, 30
genes = [f"g{i}" for i in range(n_genes)]
down = genes[:39]
up = genes[39:42]

mu = rng.normal(5, 2, n_genes)           # background expression levels (log scale)
mu[:39] = 9.5                            # "down" genes: highly expressed when healthy
mu[39:42] = 6.0                          # "up" genes: modest when healthy

def make(shift_down, shift_up, n):
    m = mu.copy()
    m[:39] += shift_down
    m[39:42] += shift_up
    return m[:, None] + rng.normal(0, 0.7, (n_genes, n))

healthy = make(0.0, 0.0, n_per_group)
damaged = make(-1.5, +3.0, n_per_group)
expr = pd.DataFrame(np.hstack([healthy, damaged]), index=genes,
                    columns=[f"h{i}" for i in range(n_per_group)] + [f"d{i}" for i in range(n_per_group)])

ds = pds_score.pds_score(expr, up, down, scale=True)
h, d = ds[:n_per_group], ds[n_per_group:]
print("finite:", bool(np.isfinite(ds).all()))
print("mean DS healthy:", round(h.mean(), 2), "| mean DS damaged:", round(d.mean(), 2))
assert d.mean() > h.mean() + 1, "score does not go up with damage"
print("TOY TEST PASSED: higher score = more damage")

finite: True
mean DS healthy: -0.98 | mean DS damaged: 0.98
TOY TEST PASSED: higher score = more damage


## 13. Simulation: depletion alone vs damage alone (invented data, no real samples)
Bulk tissue = fraction f of podocytes + (1-f) of other cells (mixed in linear space, then log2). Three groups of 20: A = healthy, 20% podocytes; B = DEPLETION only (8% podocytes, healthy cells); C = DAMAGE only (20% podocytes, but their "down" genes are reduced and "up" genes raised). Scores are computed on all 60 together. If B and C are both clearly above A, the bulk score cannot tell depletion from damage on its own.

In [14]:
rng = np.random.default_rng(20261106)
n_genes = 2000
genes = [f"g{i}" for i in range(n_genes)]
down, up = genes[:39], genes[39:42]

mu_other = rng.normal(5, 2, n_genes)          # log2 expression in non-podocyte cells
mu_pod = mu_other.copy()                      # log2 expression in healthy podocytes
mu_pod[:39] = 14                              # "down" genes: podocyte-specific, very high
mu_other[:39] = 1                             # ...and nearly absent elsewhere
mu_pod[39:42] = 6; mu_other[39:42] = 6        # "up" genes: modest everywhere
pod = 2.0 ** mu_pod
other = 2.0 ** mu_other

def simulate(f, n, damaged=False):
    p = pod.copy()
    if damaged:
        p[:39] *= 0.4                         # damaged podocytes lose their identity genes
        p[39:42] *= 4.0                       # ...and switch on the "up" genes
    cols = []
    for _ in range(n):
        fi = f * rng.uniform(0.9, 1.1)        # small natural variation in podocyte fraction
        bulk = fi * p + (1 - fi) * other
        bulk = bulk * rng.lognormal(0, 0.3, n_genes)   # sample-to-sample noise
        cols.append(np.log2(bulk + 1))
    return np.array(cols).T

A = simulate(0.20, 20)                        # healthy
B = simulate(0.08, 20)                        # depletion only
C = simulate(0.20, 20, damaged=True)          # damage only
names = [f"A{i}" for i in range(20)] + [f"B{i}" for i in range(20)] + [f"C{i}" for i in range(20)]
expr = pd.DataFrame(np.hstack([A, B, C]), index=genes, columns=names)

ds = pds_score.pds_score(expr, up, down, scale=True)
grp = pd.Series([n[0] for n in names], index=names)
print(ds.groupby(grp).mean().round(2).rename({"A": "A healthy", "B": "B depletion only", "C": "C damage only"}))
print()
print("B above A by > 0.5:", bool(ds[grp == "B"].mean() > ds[grp == "A"].mean() + 0.5))
print("C above A by > 0.5:", bool(ds[grp == "C"].mean() > ds[grp == "A"].mean() + 0.5))

A healthy          -1.32
B depletion only    0.66
C damage only       0.66
dtype: float64

B above A by > 0.5: True
C above A by > 0.5: True


## 14. Simulation benchmark: two podocyte-fraction estimators, with and without damage
Adds 20 podocyte marker genes that are NOT in the PDS gene list. Estimator A (marker-based): mean marker signal in the bulk divided by the same in a podocyte reference. Estimator B (reference-based): non-negative least squares fit of the bulk to a podocyte profile and an "other cells" profile, with the PDS genes removed. The reference profiles are noisy copies of the truth (like a real atlas). Tested on true fractions 0.03-0.25, in healthy and in damaged podocytes (damage also lowers the markers to 70%). Reports bias (mean estimate minus truth) and mean absolute error (MAE), in fraction units.

In [16]:
from scipy.optimize import nnls

rng = np.random.default_rng(20261106)
n_genes = 2000
genes = [f"g{i}" for i in range(n_genes)]
down, up, markers = genes[:39], genes[39:42], genes[42:62]
pds_set = set(down + up)

mu_other = rng.normal(5, 2, n_genes)
mu_pod = mu_other.copy()
mu_pod[:39] = 14;   mu_other[:39] = 1          # PDS "down" genes: podocyte-specific
mu_pod[39:42] = 6;  mu_other[39:42] = 6        # PDS "up" genes
mu_pod[42:62] = 12; mu_other[42:62] = 2        # 20 podocyte markers, NOT in the PDS
pod, other = 2.0 ** mu_pod, 2.0 ** mu_other

# what the researcher has: a noisy reference (like a single-nucleus atlas)
ref_pod = pod * rng.lognormal(0, 0.3, n_genes)
ref_other = other * rng.lognormal(0, 0.3, n_genes)
keep = np.array([g not in pds_set for g in genes])        # PDS genes removed from estimator B
mk = np.array([g in set(markers) for g in genes])

def simulate_linear(f_true, damaged=False):
    p = pod.copy()
    if damaged:
        p[:39] *= 0.4; p[39:42] *= 4.0                    # PDS genes change
        p[42:62] *= 0.7                                   # markers partly reduced too
    cols = []
    for f in f_true:
        bulk = f * p + (1 - f) * other
        cols.append(bulk * rng.lognormal(0, 0.3, n_genes))
    return np.array(cols).T

def est_marker(bulk):
    return bulk[mk].mean(axis=0) / ref_pod[mk].mean()

def est_nnls(bulk):
    X = np.column_stack([ref_pod[keep], ref_other[keep]])
    out = []
    for j in range(bulk.shape[1]):
        c, _ = nnls(X, bulk[keep, j])
        out.append(c[0] / (c[0] + c[1]))
    return np.array(out)

f_true = np.linspace(0.03, 0.25, 40)
rows = []
for label, dmg in [("healthy podocytes", False), ("damaged podocytes", True)]:
    bulk = simulate_linear(f_true, dmg)
    for name, fn in [("A marker-based", est_marker), ("B reference NNLS", est_nnls)]:
        est = fn(bulk)
        rows.append({"podocytes": label, "estimator": name,
                     "bias": (est - f_true).mean(),
                     "MAE": np.abs(est - f_true).mean()})
res = pd.DataFrame(rows).round(3)
print(res.to_string(index=False))
print("\nPlan tolerance: error within 0.10 (10 percentage points of the true fraction)")

        podocytes        estimator   bias   MAE
healthy podocytes   A marker-based  0.000 0.007
healthy podocytes B reference NNLS  0.015 0.023
damaged podocytes   A marker-based -0.042 0.042
damaged podocytes B reference NNLS -0.024 0.027

Plan tolerance: error within 0.10 (10 percentage points of the true fraction)


## 15. Simulation: does the mediation analysis recover a KNOWN depletion/damage split?
Three scenarios with known truth. TRUTH = (score shift caused by depletion alone) / (total score shift), measured with large counterfactual simulations. ESTIMATE = the real pipeline on 30 control + 30 disease samples: score and estimated podocyte abundance (logit), then proportion mediated = (c - c')/c with a stratified bootstrap 95% CI and the pre-registered classification. 20 simulated datasets per scenario, both abundance estimators. Bootstrap uses 500 resamples here (the real analysis uses 10,000).

In [17]:
assert "pod" in globals() and "est_nnls" in globals(), "Run section 14 first (same kernel)"
rng2 = np.random.default_rng(20261106 + 15)

def simulate_s(f_vec, s):
    """s = damage strength 0..1 (s=1 equals the damaged scenario of section 14)."""
    p = pod.copy()
    p[:39] *= (1 - 0.6 * s); p[39:42] *= (1 + 3 * s); p[42:62] *= (1 - 0.3 * s)
    cols = [(f * p + (1 - f) * other) * rng2.lognormal(0, 0.3, n_genes) for f in f_vec]
    return np.array(cols).T

def draw_f(center, n):
    return center * rng2.uniform(0.8, 1.2, n)

def to_score(bulk):
    expr = pd.DataFrame(np.log2(bulk + 1), index=genes)
    return pds_score.pds_score(expr, up, down, scale=False).to_numpy()

def true_share(f_c, f_d, s, n=150):
    ctrl = to_score(simulate_s(draw_f(f_c, n), 0)).mean()
    dep = to_score(simulate_s(draw_f(f_d, n), 0)).mean()      # depletion only
    tot = to_score(simulate_s(draw_f(f_d, n), s)).mean()      # depletion + damage
    return (dep - ctrl) / (tot - ctrl)

def ols(X, y):
    return np.linalg.lstsq(X, y, rcond=None)[0]

def share(score, m, g):
    c = ols(np.column_stack([np.ones_like(g), g]), score)[1]
    cp = ols(np.column_stack([np.ones_like(g), g, m]), score)[1]
    return c, (c - cp) / c

def boot(score, m, g, B=500):
    i0, i1 = np.where(g == 0)[0], np.where(g == 1)[0]
    cs, ps = [], []
    for _ in range(B):
        i = np.concatenate([rng2.choice(i0, len(i0)), rng2.choice(i1, len(i1))])
        c, p = share(score[i], m[i], g[i]); cs.append(c); ps.append(p)
    return np.percentile(cs, [2.5, 97.5]), np.percentile(ps, [2.5, 97.5])

def classify(lo, hi):
    if lo >= 0.5: return "mostly depletion"
    if hi <= 0.25: return "mostly damage"
    if lo >= 0.25 and hi <= 0.5: return "mixed"
    return "inconclusive"

scenarios = {"depletion only": (0.20, 0.08, 0.0),
             "mixed":           (0.20, 0.13, 0.6),
             "damage dominant": (0.20, 0.19, 1.0)}
n_grp, R = 30, 20
g = np.r_[np.zeros(n_grp), np.ones(n_grp)]
rows = []
for sc, (f_c, f_d, s) in scenarios.items():
    truth = true_share(f_c, f_d, s)
    for r in range(R):
        bulk = np.hstack([simulate_s(draw_f(f_c, n_grp), 0), simulate_s(draw_f(f_d, n_grp), s)])
        score = to_score(bulk)
        for name, fn in [("A marker", est_marker), ("B NNLS", est_nnls)]:
            f_hat = np.clip(fn(bulk), 0.005, 0.6)
            m = np.log(f_hat / (1 - f_hat))
            c, pm = share(score, m, g)
            (c_lo, c_hi), (lo, hi) = boot(score, m, g)
            rows.append({"scenario": sc, "estimator": name, "truth": truth, "estimate": pm,
                         "covers": lo <= truth <= hi, "class": classify(lo, hi),
                         "total_effect_ok": (c_lo > 0) or (c_hi < 0)})
res = pd.DataFrame(rows)
summary = res.groupby(["scenario", "estimator"]).agg(
    truth=("truth", "first"), median_estimate=("estimate", "median"),
    CI_covers_truth=("covers", "mean"),
    most_common_class=("class", lambda x: x.value_counts().index[0]),
    share_with_that_class=("class", lambda x: x.value_counts().iloc[0] / len(x))).round(2)
print(summary.to_string())

                           truth  median_estimate  CI_covers_truth most_common_class  share_with_that_class
scenario        estimator                                                                                  
damage dominant A marker    0.02             0.31             0.00      inconclusive                   1.00
                B NNLS      0.02             0.21             0.00      inconclusive                   0.80
depletion only  A marker    1.06             0.83             0.60  mostly depletion                   0.80
                B NNLS      1.06             0.49             0.00      inconclusive                   1.00
mixed           A marker    0.36             0.55             0.30      inconclusive                   0.80
                B NNLS      0.36             0.38             0.95      inconclusive                   0.95


## 16. SIMEX correction for noisy abundance, re-tested on the same simulated scenarios
Measures how noisy each abundance estimator is (SD of the error on the logit scale, in healthy simulated samples). SIMEX then adds extra noise (variance multiplied by 0.5, 1, 1.5, 2), recomputes the proportion mediated at each level, fits a quadratic curve and extrapolates to -1 (= no noise). Compares the naive estimate with the SIMEX estimate against the known truth. 10 datasets per scenario, 200 bootstrap resamples (the real analysis uses 10,000).

In [18]:
def sigma_u(fn):
    f = np.linspace(0.05, 0.25, 60)
    est = np.clip(fn(simulate_s(f, 0)), 0.005, 0.6)
    return (np.log(est / (1 - est)) - np.log(f / (1 - f))).std(ddof=1)

su = {"A marker": sigma_u(est_marker), "B NNLS": sigma_u(est_nnls)}
print("Estimator noise on logit scale (SD):", {k: round(v, 3) for k, v in su.items()})

def share_simex(score, m, g, s_u, lambdas=(0.5, 1, 1.5, 2), reps=10):
    pts = [share(score, m, g)[1]]
    for lam in lambdas:
        vals = [share(score, m + rng2.normal(0, np.sqrt(lam) * s_u, len(m)), g)[1]
                for _ in range(reps)]
        pts.append(np.mean(vals))
    coef = np.polyfit(np.r_[0, lambdas], pts, 2)
    return np.polyval(coef, -1.0)

def boot_simex(score, m, g, s_u, B=200):
    i0, i1 = np.where(g == 0)[0], np.where(g == 1)[0]
    ps = []
    for _ in range(B):
        i = np.concatenate([rng2.choice(i0, len(i0)), rng2.choice(i1, len(i1))])
        ps.append(share_simex(score[i], m[i], g[i], s_u))
    return np.percentile(ps, [2.5, 97.5])

R2, rows2 = 10, []
for sc, (f_c, f_d, s) in scenarios.items():
    truth = true_share(f_c, f_d, s)
    for r in range(R2):
        bulk = np.hstack([simulate_s(draw_f(f_c, n_grp), 0), simulate_s(draw_f(f_d, n_grp), s)])
        score = to_score(bulk)
        for name, fn in [("A marker", est_marker), ("B NNLS", est_nnls)]:
            f_hat = np.clip(fn(bulk), 0.005, 0.6)
            m = np.log(f_hat / (1 - f_hat))
            naive = share(score, m, g)[1]
            sim = share_simex(score, m, g, su[name])
            lo, hi = boot_simex(score, m, g, su[name])
            rows2.append({"scenario": sc, "estimator": name, "truth": truth,
                          "naive": naive, "simex": sim,
                          "covers": lo <= truth <= hi, "class": classify(lo, hi)})
res2 = pd.DataFrame(rows2)
out = res2.groupby(["scenario", "estimator"]).agg(
    truth=("truth", "first"), naive_median=("naive", "median"), simex_median=("simex", "median"),
    CI_covers=("covers", "mean"),
    class_=("class", lambda x: x.value_counts().index[0])).round(2)
print(out.to_string())

Estimator noise on logit scale (SD): {'A marker': np.float64(0.076), 'B NNLS': np.float64(0.182)}
                           truth  naive_median  simex_median  CI_covers            class_
scenario        estimator                                                                
damage dominant A marker    0.02          0.31          0.41        0.0      inconclusive
                B NNLS      0.02          0.18          0.35        0.0      inconclusive
depletion only  A marker    1.02          0.77          0.98        0.7  mostly depletion
                B NNLS      1.02          0.47          0.90        0.9  mostly depletion
mixed           A marker    0.35          0.59          0.72        0.0      inconclusive
                B NNLS      0.35          0.38          0.70        0.2      inconclusive


## 17. Simulation: false-alarm rate and power of the rule "depletion alone is rejected"
Reruns the three scenarios with SIMEX and keeps the bootstrap interval. Rule: depletion alone is rejected if the upper limit of the 95% interval for the depletion share is below 1.0 (or below 0.9). Depletion-only row = FALSE-ALARM rate (want low). Mixed and damage-dominant rows = POWER (want high).

In [19]:
rows3, R3 = [], 20
for sc, (f_c, f_d, s) in scenarios.items():
    truth = true_share(f_c, f_d, s)
    for r in range(R3):
        bulk = np.hstack([simulate_s(draw_f(f_c, n_grp), 0), simulate_s(draw_f(f_d, n_grp), s)])
        score = to_score(bulk)
        for name, fn in [("A marker", est_marker), ("B NNLS", est_nnls)]:
            f_hat = np.clip(fn(bulk), 0.005, 0.6)
            m = np.log(f_hat / (1 - f_hat))
            est = share_simex(score, m, g, su[name])
            lo, hi = boot_simex(score, m, g, su[name])
            rows3.append({"scenario": sc, "estimator": name, "truth": truth,
                          "simex": est, "hi": hi})
res3 = pd.DataFrame(rows3)
tab = res3.groupby(["scenario", "estimator"]).agg(
    truth=("truth", "first"),
    median_simex=("simex", "median"),
    median_upper=("hi", "median"),
    reject_if_upper_below_1=("hi", lambda x: (x < 1.0).mean()),
    reject_if_upper_below_0_9=("hi", lambda x: (x < 0.9).mean())).round(2)
print(tab.to_string())

                           truth  median_simex  median_upper  reject_if_upper_below_1  reject_if_upper_below_0_9
scenario        estimator                                                                                       
damage dominant A marker    0.03          0.40          0.59                     1.00                       1.00
                B NNLS      0.03          0.38          0.59                     0.95                       0.95
depletion only  A marker    0.98          0.87          1.29                     0.05                       0.00
                B NNLS      0.98          0.79          1.23                     0.15                       0.05
mixed           A marker    0.37          0.73          1.02                     0.45                       0.20
                B NNLS      0.37          0.66          0.99                     0.55                       0.25
